# Exp 3 - Throughput and Response-Time Analysis in Autonomous Communication

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Estimate throughput and response time for autonomous communication packets.

Throughput tells how much payload is delivered per second. Response time tells how long each message takes from generation to completion. A communication link can have high throughput but still be unsuitable for control if response time is too large or too variable.

## Textbook Notes and Case Studies

### 1. Textbook Background

Throughput measures how much useful data is delivered per unit time. Response time measures how long a request takes from arrival to completion. In autonomous communication, high throughput is useful only if the response time still satisfies the control deadline.

A common mistake is to optimize only bandwidth. A system can have high throughput while still being unsuitable for real-time control if queues grow and response time becomes unpredictable. For safety-related communication, bounded delay and bounded queueing are usually more important than peak transfer rate.

### 2. Architecture Notes

```
Request Arrivals -> Input Queue -> Communication/Processing Server -> Completed Responses
       |                 |                      |
       v                 v                      v
Arrival Rate        Queue Length           Service Rate
       |                                        |
       +----------- Response-Time Log ----------+
```

The server may represent a network link, gateway, edge node, or processing unit. As arrival rate approaches service rate, queueing delay increases sharply. This is why overload detection is important in autonomous and industrial systems.

### 3. Important Formulas

Throughput:

```
throughput = total_data_delivered / total_time
```

Request response time:

```
response_time = completion_time - arrival_time
```

Server utilization:

```
utilization = arrival_rate / service_rate
```

For a simple M/M/1 queue model, expected response time increases as utilization approaches 1:

```
expected_response_time = 1 / (service_rate - arrival_rate)
```

This formula is a simplified model, not a full proof of real-time safety. It is useful for intuition: small capacity margins can create large delays.

### 4. Classroom Case Studies

Case Study A - Autonomous Sensor Gateway:
Multiple sensors send lidar, radar, and camera summaries to a central controller. If the gateway accepts more data than it can forward, queues form and response time rises. A throughput-only result may look good while urgent messages are delayed behind large non-critical messages.

Case Study B - Intersection Infrastructure:
Roadside units collect vehicle messages and broadcast signal-phase information. During congestion, arrival rate increases. A robust design uses prioritization or rate limiting so safety messages do not wait behind routine telemetry.

Case Study C - Edge Video Analytics:
An edge node receives frames for object detection. Higher frame rate increases throughput demand. If response time exceeds the frame usefulness window, the detector may report objects too late.

### 5. Analysis Checklist

Include offered load, delivered throughput, response time, queue trend, and overload condition. A good post-lab answer explains where saturation begins and what design action would reduce delay: filtering, compression, priority queues, load shedding, or more processing capacity.

### 6. Source Notes

- Python timing and measurement support: https://docs.python.org/3/library/time.html
- IEEE Time-Sensitive Networking overview for deterministic communication context: https://1.ieee802.org/tsn/


## Architecture

```text
Packet Workload
  |-- packet size
  |-- link rate
  |-- queue delay
  |-- processing delay
          |
          v
Transmission Model
  |-- transmission time = packet bits / link rate
          |
          v
Response-Time Model
  |-- response time = transmission + queueing + processing
          |
          v
Throughput Calculator
  |-- total delivered bits / elapsed time
```

The simulation separates link capacity from queueing delay because congestion often shows up first as response-time growth.

## Formulas and Required Theory

\[
\text{transmission time} = \frac{\text{packet size in bits}}{\text{link rate in bits/s}}
\]

\[
\text{response time} = T_{transmission} + T_{queueing} + T_{processing}
\]

\[
\text{throughput} = \frac{\text{total delivered bits}}{\text{total elapsed time}}
\]

When offered load increases, queueing delay usually grows faster than raw transmission time. This is why autonomous systems need both bandwidth planning and latency monitoring.

## In-Lab Method

1. Generate packets with different payload sizes.
2. Compute transmission time from packet size and link rate.
3. Add queueing and processing delays.
4. Measure total delivered bits and total elapsed time.
5. Report throughput, mean response time, and maximum response time.

In [1]:
import random
import statistics

rng = random.Random(341403)
packets = []
clock = 0.0
for i in range(40):
    size_bytes = rng.choice([128, 256, 512, 1024, 1400])
    link_mbps = 12
    tx_ms = (size_bytes * 8) / (link_mbps * 1000)
    queue_ms = rng.uniform(0.2, 3.8)
    processing_ms = rng.uniform(0.4, 2.5)
    response_ms = tx_ms + queue_ms + processing_ms
    clock += response_ms
    packets.append((i + 1, size_bytes, response_ms))

total_bits = sum(p[1] * 8 for p in packets)
duration_s = clock / 1000
throughput_mbps = total_bits / duration_s / 1_000_000
responses = [p[2] for p in packets]

print("EXP 3 - IN-LAB RESULT")
print(f"Packets analysed      : {len(packets)}")
print(f"Total payload         : {total_bits / 8:.0f} bytes")
print(f"Measured throughput   : {throughput_mbps:.3f} Mbps")
print(f"Mean response time    : {statistics.mean(responses):.2f} ms")
print(f"Max response time     : {max(responses):.2f} ms")
print("First five packets:", packets[:5])

EXP 3 - IN-LAB RESULT
Packets analysed      : 40
Total payload         : 31264 bytes
Measured throughput   : 1.541 Mbps
Mean response time    : 4.06 ms
Max response time     : 6.83 ms
First five packets: [(1, 512, 2.220407117623182), (2, 128, 3.795172099694435), (3, 1024, 2.431030740592837), (4, 1024, 5.554517512164553), (5, 1400, 2.8858043128709827)]


## Post-Lab Method

The post-lab cell repeats the experiment under different load multipliers. It shows how response time increases when the communication system becomes busier.

In [2]:
import random
import statistics

def trial(load, seed=341430):
    rng = random.Random(seed)
    responses = []
    bits = 0
    elapsed = 0.0
    for _ in range(100):
        size = rng.choice([256, 512, 1024, 1400])
        link_mbps = 12
        tx = (size * 8) / (link_mbps * 1000)
        queue = rng.uniform(0.3, 3.0) * load
        proc = rng.uniform(0.4, 2.2)
        rt = tx + queue + proc
        responses.append(rt)
        elapsed += rt
        bits += size * 8
    return bits / (elapsed / 1000) / 1_000_000, statistics.mean(responses), max(responses)

print("EXP 3 - POST-LAB LOAD EFFECT")
print(f"{'Load':>6} {'Throughput Mbps':>16} {'Mean RT ms':>12} {'Max RT ms':>10}")
for load in [0.5, 0.8, 1.0, 1.3, 1.6]:
    throughput, mean_rt, max_rt = trial(load)
    print(f"{load:6.1f} {throughput:16.3f} {mean_rt:12.2f} {max_rt:10.2f}")

EXP 3 - POST-LAB LOAD EFFECT
  Load  Throughput Mbps   Mean RT ms  Max RT ms
   0.5            2.634         2.66       4.23
   0.8            2.235         3.13       5.05
   1.0            2.030         3.45       5.60
   1.3            1.785         3.92       6.43
   1.6            1.592         4.39       7.25


## What to Write in the Lab Record

- Record throughput in Mbps.
- Record mean and maximum response time.
- Explain why maximum response time matters for autonomous control.
- Compare how load changes throughput and delay.

## References

- Python `time` module documentation: https://docs.python.org/3/library/time.html
- Python `statistics` module documentation: https://docs.python.org/3/library/statistics.html
- Python `random` module documentation: https://docs.python.org/3/library/random.html